In [10]:
!pip install pandas pillow

In [4]:
# import pandas as pd
# import cv2
# import os

# base_path = "/workspace/small_dataset2/small_dataset"
# label_file = os.path.join(base_path, "labels.csv")
# output_dir = "prep"

# os.makedirs(output_dir, exist_ok=True)

# df = pd.read_csv(label_file, names=["img_num", "rel_x", "rel_y", "cell_roi_w", "cell_roi_h", "label"])

# normal_classes = ['seg_neutrophil', 'lymphocyte', 'band_neutrophil', 'eosinophil', 'monocyte', 'basophil']
# prep_data = []

In [5]:
# for index, row in df.iterrows():

#     img_num = int(row['img_num'])

#     img_path = os.path.join(base_path, f"{img_num}_7.jpg")
#     img = cv2.imread(img_path)

#     left = int(row['rel_x'])
#     top = int(row['rel_y'])
#     right = int(left + row['cell_roi_w'])
#     bottom = int(top + row['cell_roi_h'])

#     cropped_img = img[top:bottom, left:right]

#     new_filename = f"cropped{img_num}.jpg"
#     save_path = os.path.join(output_dir, new_filename)

#     cv2.imwrite(save_path, cropped_img)

#     label = row['label']
#     new_label = 'normal' if label in normal_classes else label

#     prep_data.append([new_filename, new_label])

In [8]:
# prep_df = pd.DataFrame(prep_data, columns=['filename', 'label'])
# prep_df.to_csv(os.path.join(output_dir, 'prep_label.csv'), index=False, header=False)

In [10]:
import pandas as pd
import cv2
import os

base_path = "/workspace/small_dataset2/small_dataset"
label_file = os.path.join(base_path, "labels.csv")
output_dir = "prep"

os.makedirs(output_dir, exist_ok=True)

# Read the CSV
df = pd.read_csv(label_file, names=["img_num", "rel_x", "rel_y", "cell_roi_w", "cell_roi_h", "label"])

normal_classes = ['seg_neutrophil', 'lymphocyte', 'eosinophil', 'monocyte', 'basophil']

# 1. Convert to strict binary: 'normal' vs 'abnormal'
df['binary_label'] = df['label'].apply(lambda x: 'normal' if x in normal_classes else 'abnormal')

print("--- Distribution Before Balancing ---")
print(df['binary_label'].value_counts())

# 2. Separate into the two classes
normal_df = df[df['binary_label'] == 'normal']
abnormal_df = df[df['binary_label'] == 'abnormal']

# 3. Find the smallest class size to ensure perfect 1:1 balance
# (You can also replace 'min_size' with a hardcoded number like 500 if you want an even smaller dataset)
min_size = min(len(normal_df), len(abnormal_df))

print(f"\nShrinking both classes to exactly {min_size} samples each...")

# 4. Undersample BOTH classes to match the min_size
normal_df_balanced = normal_df.sample(n=min_size, random_state=42)
abnormal_df_balanced = abnormal_df.sample(n=min_size, random_state=42)

# Recombine and shuffle the dataframe
balanced_df = pd.concat([normal_df_balanced, abnormal_df_balanced]).sample(frac=1, random_state=42).reset_index(drop=True)

prep_data = []

# 5. Crop and Save
for index, row in balanced_df.iterrows():
    img_num = int(row['img_num'])
    img_path = os.path.join(base_path, f"{img_num}_7.jpg")
    img = cv2.imread(img_path)
    
    if img is None:
        print(f"Warning: Could not read {img_path}")
        continue

    left = int(row['rel_x'])
    top = int(row['rel_y'])
    right = int(left + row['cell_roi_w'])
    bottom = int(top + row['cell_roi_h'])

    cropped_img = img[top:bottom, left:right]
    final_label = row['binary_label']

    # Unique filenames to prevent overwriting
    base_filename = f"cropped_{img_num}_cell{index}.jpg"
    save_path = os.path.join(output_dir, base_filename)
    
    cv2.imwrite(save_path, cropped_img)
    prep_data.append([base_filename, final_label])

# Save the perfectly balanced label CSV
prep_df = pd.DataFrame(prep_data, columns=['filename', 'label'])
prep_df.to_csv(os.path.join(output_dir, 'prep_label.csv'), index=False, header=False)

print("\n--- Final Balanced Distribution ---")
print(prep_df['label'].value_counts())

--- Distribution Before Balancing ---
binary_label
normal      15975
abnormal     8825
Name: count, dtype: int64

Shrinking both classes to exactly 8825 samples each...

--- Final Balanced Distribution ---
label
normal      8825
abnormal    8825
Name: count, dtype: int64
